Here is a quick way to build and environment to follow the scripts through.

In [ ]:
%conda create --name sd_env python=3.13.0
%conda activate sd_env
%pip install spatialdata==0.7.3 spatialdata_plot==0.3.4

We will need `spatialdata` and `spatialdata-plot`.

In [ ]:
import os
import spatialdata as sd
import spatialdata_plot as spd

/Users/amanuky/miniforge3/envs/sd_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Accession

In [18]:
sdata = sd.read_zarr("../../extdata/blobs.zarr")
sdata

/var/folders/vf/d8kg507x41xfh6z9vgv9skksdsn29w/T/ipykernel_69219/477556509.py:1: UserWarning: SpatialData is not stored in the most current format. If you want to use Zarr v3, please write the store to a new location using `sdata.write()`.
  sdata = sd.read_zarr("../../extdata/blobs.zarr")
no parent found for <ome_zarr.reader.Label object at 0x13bb81090>: None
no parent found for <ome_zarr.reader.Label object at 0x13a974510>: None
/Users/amanuky/miniforge3/envs/sd_env/lib/python3.13/site-packages/zarr/core/group.py:3289: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


SpatialData object, with associated Zarr store: /Users/amanuky/Dropbox/Research/MDC/Projects/SpatialData/Packages/spatialdataR/inst/extdata/blobs.zarr
├── Images
│     ├── 'blobs_image': DataArray[cyx] (3, 64, 64)
│     └── 'blobs_multiscale_image': DataTree[cyx] (3, 64, 64), (3, 32, 32), (3, 16, 16)
├── Labels
│     ├── 'blobs_labels': DataArray[yx] (64, 64)
│     └── 'blobs_multiscale_labels': DataTree[yx] (64, 64), (32, 32), (16, 16)
├── Points
│     └── 'blobs_points': DataFrame with shape: (<Delayed>, 4) (2D points)
├── Shapes
│     ├── 'blobs_circles': GeoDataFrame shape: (5, 2) (2D shapes)
│     ├── 'blobs_multipolygons': GeoDataFrame shape: (2, 1) (2D shapes)
│     └── 'blobs_polygons': GeoDataFrame shape: (5, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (10, 3)
with coordinate systems:
    ▸ 'affine', with elements:
        blobs_labels (Labels)
    ▸ 'global', with elements:
        blobs_image (Images), blobs_multiscale_image (Images), blobs_labels (Labels), blobs_mu

In [4]:
sdata.images["blobs_image"]
sdata.get("blobs_image")
sdata["blobs_image"]

<xarray.DataArray 'image' (c: 3, y: 64, x: 64)> Size: 98kB
dask.array<from-zarr, shape=(3, 64, 64), dtype=float64, chunksize=(3, 64, 64), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) int64 24B 0 1 2
  * y        (y) float64 512B 0.5 1.5 2.5 3.5 4.5 ... 59.5 60.5 61.5 62.5 63.5
  * x        (x) float64 512B 0.5 1.5 2.5 3.5 4.5 ... 59.5 60.5 61.5 62.5 63.5
Attributes:
    transform:  {'global': Identity }

In [15]:
from spatialdata import concatenate
concatenate({"s1": sdata, "s2": sdata})

SpatialData object
├── Images
│     ├── 'blobs_image-s1': DataArray[cyx] (3, 64, 64)
│     ├── 'blobs_image-s2': DataArray[cyx] (3, 64, 64)
│     ├── 'blobs_multiscale_image-s1': DataTree[cyx] (3, 64, 64), (3, 32, 32), (3, 16, 16)
│     └── 'blobs_multiscale_image-s2': DataTree[cyx] (3, 64, 64), (3, 32, 32), (3, 16, 16)
├── Labels
│     ├── 'blobs_labels-s1': DataArray[yx] (64, 64)
│     ├── 'blobs_labels-s2': DataArray[yx] (64, 64)
│     ├── 'blobs_multiscale_labels-s1': DataTree[yx] (64, 64), (32, 32), (16, 16)
│     └── 'blobs_multiscale_labels-s2': DataTree[yx] (64, 64), (32, 32), (16, 16)
├── Points
│     ├── 'blobs_points-s1': DataFrame with shape: (<Delayed>, 4) (2D points)
│     └── 'blobs_points-s2': DataFrame with shape: (<Delayed>, 4) (2D points)
├── Shapes
│     ├── 'blobs_circles-s1': GeoDataFrame shape: (5, 2) (2D shapes)
│     ├── 'blobs_circles-s2': GeoDataFrame shape: (5, 2) (2D shapes)
│     ├── 'blobs_multipolygons-s1': GeoDataFrame shape: (2, 1) (2D shapes)
│     ├─

In [ ]:
sdata.labels["blobs_labels"].attrs

{'transform': {'global': Identity }}

# Coord. Systems and transformations

In [5]:
sdata.coordinate_systems

['sequence', 'global', 'affine', 'scale', 'translation']

In [ ]:
from spatialdata.transformations import get_transformation
get_transformation(sdata.images["blobs_image"])


Identity 

In [ ]:
from spatialdata.transformations import get_transformation
get_transformation(sdata.images["blobs_image"], get_all=True)


{'global': Identity }

In [31]:
from spatialdata import transform
from spatialdata.transformations import Affine
import math
theta = math.pi / 6
rotation = Affine(
    [
        [math.cos(theta), -math.sin(theta), 0],
        [math.sin(theta), math.cos(theta), 0],
        [0, 0, 1],
    ],
    input_axes=("x", "y"),
    output_axes=("x", "y"),
)
print(sdata)
transform(sdata.labels["blobs_labels"], to_coordinate_system="scale")

SpatialData object, with associated Zarr store: /Users/amanuky/Dropbox/Research/MDC/Projects/SpatialData/Packages/spatialdataR/inst/extdata/blobs.zarr
├── Images
│     ├── 'blobs_image': DataArray[cyx] (3, 64, 64)
│     └── 'blobs_multiscale_image': DataTree[cyx] (3, 64, 64), (3, 32, 32), (3, 16, 16)
├── Labels
│     ├── 'blobs_labels': DataArray[yx] (64, 64)
│     └── 'blobs_multiscale_labels': DataTree[yx] (64, 64), (32, 32), (16, 16)
├── Points
│     └── 'blobs_points': DataFrame with shape: (<Delayed>, 4) (2D points)
├── Shapes
│     ├── 'blobs_circles': GeoDataFrame shape: (5, 2) (2D shapes)
│     ├── 'blobs_multipolygons': GeoDataFrame shape: (2, 1) (2D shapes)
│     └── 'blobs_polygons': GeoDataFrame shape: (5, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (10, 3)
with coordinate systems:
    ▸ 'affine', with elements:
        blobs_labels (Labels)
    ▸ 'global', with elements:
        blobs_image (Images), blobs_multiscale_image (Images), blobs_labels (Labels), blobs_mu

/Users/amanuky/miniforge3/envs/sd_env/lib/python3.13/site-packages/zarr/core/group.py:3289: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


<xarray.DataArray 'image' (y: 192, x: 128)> Size: 49kB
dask.array<affine_transform, shape=(192, 128), dtype=int16, chunksize=(64, 64), chunktype=numpy.ndarray>
Coordinates:
  * y        (y) float64 2kB 0.5 1.5 2.5 3.5 4.5 ... 188.5 189.5 190.5 191.5
  * x        (x) float64 1kB 0.5 1.5 2.5 3.5 4.5 ... 124.5 125.5 126.5 127.5
Attributes:
    transform:  {'scale': Translation (y, x)\n    [0. 0.]}

# Operations

In [24]:
from spatialdata import get_extent
# get_extent(sdata)
get_extent(sdata.images["blobs_image"])

{'y': (np.float64(0.0), np.float64(64.0)),
 'x': (np.float64(0.0), np.float64(64.0))}

In [32]:
sdata

/Users/amanuky/miniforge3/envs/sd_env/lib/python3.13/site-packages/zarr/core/group.py:3289: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


SpatialData object, with associated Zarr store: /Users/amanuky/Dropbox/Research/MDC/Projects/SpatialData/Packages/spatialdataR/inst/extdata/blobs.zarr
├── Images
│     ├── 'blobs_image': DataArray[cyx] (3, 64, 64)
│     └── 'blobs_multiscale_image': DataTree[cyx] (3, 64, 64), (3, 32, 32), (3, 16, 16)
├── Labels
│     ├── 'blobs_labels': DataArray[yx] (64, 64)
│     └── 'blobs_multiscale_labels': DataTree[yx] (64, 64), (32, 32), (16, 16)
├── Points
│     └── 'blobs_points': DataFrame with shape: (<Delayed>, 4) (2D points)
├── Shapes
│     ├── 'blobs_circles': GeoDataFrame shape: (5, 2) (2D shapes)
│     ├── 'blobs_multipolygons': GeoDataFrame shape: (2, 1) (2D shapes)
│     └── 'blobs_polygons': GeoDataFrame shape: (5, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (10, 3)
with coordinate systems:
    ▸ 'affine', with elements:
        blobs_labels (Labels)
    ▸ 'global', with elements:
        blobs_image (Images), blobs_multiscale_image (Images), blobs_labels (Labels), blobs_mu

In [35]:
sdata_cropped = sdata.query.bounding_box(
    min_coordinate=[0, 0],
    max_coordinate=[30, 30],
    axes=("x", "y"),
    target_coordinate_system="global",
)
sdata_cropped
get_extent(sdata_cropped)

/Users/amanuky/miniforge3/envs/sd_env/lib/python3.13/functools.py:929: UserWarning: The object has `points` element. Depending on the number of points, querying MAY suffer from performance issues. Please consider filtering the object before calling this function by calling the `subset()` method of `SpatialData`.
  return dispatch(args[0].__class__)(*args, **kw)


{'y': (np.float64(0.0), np.float64(35.211751674196506)),
 'x': (np.float64(0.0), np.float64(35.688261562846535))}

In [37]:
from spatialdata import bounding_box_query
bounding_box_query(
    sdata.images["blobs_image"],
    min_coordinate=[0, 0],
    max_coordinate=[30, 30],
    axes=("x", "y"),
    target_coordinate_system="global",
)

<xarray.DataArray 'image' (c: 3, y: 30, x: 30)> Size: 22kB
dask.array<getitem, shape=(3, 30, 30), dtype=float64, chunksize=(3, 30, 30), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) int64 24B 0 1 2
  * y        (y) float64 240B 0.5 1.5 2.5 3.5 4.5 ... 25.5 26.5 27.5 28.5 29.5
  * x        (x) float64 240B 0.5 1.5 2.5 3.5 4.5 ... 25.5 26.5 27.5 28.5 29.5
Attributes:
    transform:  {'global': Identity }

# Others

In [20]:
sdata.aggregate(values="blobs_image", by="blobs_labels", agg_func="mean")

SpatialData object
├── Labels
│     └── 'blobs_labels': DataArray[yx] (64, 64)
└── Tables
      └── 'table': AnnData (10, 3)
with coordinate systems:
    ▸ 'affine', with elements:
        blobs_labels (Labels)
    ▸ 'global', with elements:
        blobs_labels (Labels)
    ▸ 'scale', with elements:
        blobs_labels (Labels)
    ▸ 'sequence', with elements:
        blobs_labels (Labels)
    ▸ 'translation', with elements:
        blobs_labels (Labels)